## tl;dr

- 得意技は**3年生5月の固定イベント**で選び、その月の通常イベント1件を置き換える。
- 成長補正は次の練習（3年生6月）から、ショート倍率の適用後に加える。同じ練習内ではプラス1回・マイナス1回まで。
- 後半発動に合わせ、狭い・解禁が遅い技ほど成長補正を大きくする。ノーマルでは能力強化8種がカードなし比で平均+8.1〜+13.0ptに収まった。
- 固定補正はイベント選択時に適用し、音はめは演技構成+2、ピルエットは全部門難易度+1、ボディ系は全部門新奇性+2とする。
- ハプニングは新規カードとして採用し、プレイヤーの審査員補正を通常の±3点から**±8点**へ広げる。平均補正は0のまま。


## Context & Methods

現行ショート版（24行動）のゲームバランスに対し、得意技カード9種を比較した。  
分析の目的は、カードによる卒業時能力と獲得ポイントへの影響を同じ価値帯へ収めつつ、専門性を残すこと。

### Key Assumptions

- 3年生5月（内部turn 26）の固定イベントでカードを選び、状態依存・ランダムを含む通常イベント1件の代わりにする。
- 「伸び+N」は、イベント後の該当練習が成功して1点以上伸びた場合に発動する。
- +Nはショート版の2倍処理後に加算する。
- 同じ練習の3枠で条件を複数回満たしても、プラス効果は1回まで。マイナス効果も1回まで。
- 「+N」とだけ指定されたカードは、3年生5月の選択時に固定能力として加える。
- カードごと・難易度ごとに、既存の有力方針群から練習方針と大会方針を再選定した。
- 乱数シードはカード間で共通。各最終評価は難易度ごとに400周。


In [ ]:
from pathlib import Path
import json

ROOT = Path.cwd()
RESULT_FILES = [
    ROOT / "docs/analysis/2026-07-23-technique-card-balance-college.json",
    ROOT / "docs/analysis/2026-07-23-technique-card-balance-highschool.json",
    ROOT / "docs/analysis/2026-07-23-technique-card-balance-juniorhigh.json",
]

rows = []
for result_file in RESULT_FILES:
    payload = json.loads(result_file.read_text(encoding="utf-8"))
    rows.extend(payload["results"])

difficulty_labels = {
    "college": "ハード",
    "highschool": "ノーマル",
    "juniorhigh": "イージー",
}
for row in rows:
    row["difficulty"] = difficulty_labels[row["background"]]
    row["points_delta"] = row["pairedPointDelta"]
    row["points_delta_ci95"] = 1.96 * row["pairedPointDeltaSe"]
    row["ability_delta"] = row["pairedAbilityDelta"]

len(rows), sorted({row["difficulty"] for row in rows})

## Data

分析対象は、現行コードの成長式・大会採点式と、3年生5月イベントを分析上で再現したシミュレーター出力。  
カード間で同じ乱数シードを使い、カードなしとの差を同一周回ごとに取った。ハプニングは±6・±8・±10を比較し、最終案には±8を採用した。


In [ ]:
recommended_ids = [
    "integral", "high_toss", "fts", "picture", "pirouette",
    "sadistic", "on_beat", "body", "happening8"
]
effect_labels = {
    "integral": "3年5月以降、1DH難易度の伸び+8",
    "high_toss": "3年5月以降、2D/3D操作の伸び+6・新奇性の伸び-1",
    "fts": "3年5月以降、3D+難易度の伸び+10",
    "picture": "3年5月以降、1DH/1DV新奇性の伸び+3",
    "pirouette": "選択時、全ジャンル難易度+1",
    "sadistic": "3年5月以降、1DH新奇性の伸び+5",
    "on_beat": "選択時、演技構成+2",
    "body": "選択時、全ジャンル新奇性+2",
    "happening8": "大会の審査員補正を±3から±8へ",
}

normal = [
    row for row in rows
    if row["background"] == "highschool" and row["card"] in recommended_ids
]
normal.sort(key=lambda row: row["points_delta"], reverse=True)

print("カード\t調整後効果\t平均pt差±95%幅\t平均能力差\t直接追加")
for row in normal:
    print(
        f'{row["label"]}\t{effect_labels[row["card"]]}\t'
        f'{row["points_delta"]:+.1f}±{row["points_delta_ci95"]:.1f}\t'
        f'{row["ability_delta"]:+.2f}\t'
        f'{row["meanTechniqueDelta"]:.1f}'
    )

## Results

ノーマルでは、ハプニングを除く能力強化8種がカードなし比で平均+8.1〜+13.0ptに収まった。  
卒業時平均能力の増分も+0.37〜+0.60で、カード間の突出は小さい。  
ハプニング±8は平均得点を増やすカードではなく、カードなしとの差の標準偏差を約80ptまで広げる逆転用カードになった。


In [ ]:
print("調整後の得意技カード別 平均獲得ポイント差（ノーマル、400周）")
scale = 2.0
for row in sorted(normal, key=lambda item: item["points_delta"]):
    width = max(0, round(row["points_delta"] / scale))
    print(f'{row["label"]:12} {"█" * width:<14} {row["points_delta"]:+5.1f}')

In [ ]:
print("難易度別比較（平均pt差 / 平均能力差）")
for card_id in recommended_ids:
    label = next(row["label"] for row in rows if row["card"] == card_id)
    values = []
    for difficulty in ["ハード", "ノーマル", "イージー"]:
        row = next(
            row for row in rows
            if row["card"] == card_id and row["difficulty"] == difficulty
        )
        values.append(f'{difficulty}:{row["points_delta"]:+.1f}±{row["points_delta_ci95"]:.1f}/{row["ability_delta"]:+.2f}')
    print(label, " | ".join(values))

## Takeaways

1. **3年生5月は育成方針を切り替える地点として機能する。** 残り約8回の練習で得意技を完成させる流れになる。
2. **後半発動の成長型には補償が必要。** 対象の狭さと解禁時期に応じて+3〜+10へ差をつけるとノーマルで同じ帯へ入る。
3. **演技構成は1点の大会価値が高い。** 音はめは選択時+2で他カードと同程度になる。
4. **ハプニングは平均強化ではなく逆転用。** ±8なら通常の±3より明確に荒れるが、±10ほど極端ではない。
5. **カード間を完全同値にはしない。** 早熟、後半型、安定、逆転という使い分けを残す。


## Caveats

- カード間で共通乱数を使ったが、400周では小さい差の厳密な順位づけには不足する。ここでは完全な序列ではなく、突出・不足の検出に使った。
- ハードは最適方針の選定が不安定。3D+が未解禁のままならFTSは効果0になるため、固定イベントでは未解禁技を候補から除外する必要がある。
- イージーでは能力上限の影響で固定加算型の価値が下がる。背景ごとに同価値にするより、育成ルートとの相性として残す案を採っている。
- ハプニングは得点平均を直接増やさないが、順位・ポイントの非線形性によって、劣勢時には期待ポイントが少し上がり得る。
